# Mount Elgon flood records — preprocessing

Cleans the DesInventar disaster inventory into a **district-day** frame suitable
for training a flood prediction model, where one row answers "was district D
flooded on day T".

**Source:** `dataset/mountelgon-desinventar.xls` — 271 rows, 17 columns,
1933–2018. DesInventar only; see section 2 for why EM-DAT was rejected.

**Study area:** seven districts — Butaleja, Mbale, Manafwa, Bududa, Sironko,
Bulambuli, Kapchorwa. BUKWO and KWEEN are excluded as high-massif,
landslide-dominated, and contributing 1–2 district-days between them.

The notebook works on a single `df`, reassigned as each step is applied:

| Step | Effect on `df` |
| --- | --- |
| 1. Load, restrict to the study area | 271 → 264 |
| 3. Drop invalid dates | 264 → 216 |
| 4. Collapse to one row per district-day | 216 → 93 |
| 5. Group into multi-district storms | (no row change) |
| 6. Expand to day spans (7-day cap) | 93 → 137 |

> Cells are order-dependent — run top to bottom. Re-running a step out of
> sequence will report against an already-cleaned frame.

## 1. Setup and load

`dataset_path` is relative to the repository root. A notebook has no `__file__`
and the kernel's working directory depends on where Jupyter was launched, so the
root is found by walking up to the directory containing `dataset/` rather than by
a fixed number of `.parents` hops.

In [344]:
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

dataset_path = "dataset/mountelgon-desinventar.xls"

DATE_COLUMN = "Date (YMD)"


pd.set_option('display.max_rows', None) # No cap on rows printed out


def find_repo_root(marker: str = "dataset") -> Path:
    """Walk up from the working directory until `marker` is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


# `dataset_path` is relative to the repository root
REPO_ROOT = find_repo_root()

path = REPO_ROOT / dataset_path
if not path.exists():
    raise FileNotFoundError(f"Dataset not found: {path}")

df = pd.read_excel(path)
print(f"loaded {len(df)} rows x {len(df.columns)} columns from {path.name}")

loaded 271 rows x 17 columns from mountelgon-desinventar.xls


### What the columns hold

Locations are recorded at three nested levels — **District > Subcounty >
Parish** — with a free-text `Location` used when `Subcounty` and `Parish` are
blank. This matters for the duplicate check in section 3: two rows sharing a
date and district are usually *different sub-locations*, not repeats.

Two columns are worth noting before any cleaning:

- **`Event` is `FLOOD` in all 271 rows**, so it carries no information and any
  "same event" grouping below reduces to a date + place comparison.
- **`Deaths` is recorded in only 17 rows** (87 deaths total) and
  `Duration (d)` in 57, so most rows describe an event without quantifying it.

In [345]:
print(df[["Date (YMD)", "Duration (d)", "District", "Subcounty", "Parish", "Location"]].head(10))

print("\nnon-null counts for the key fields:")
print(df[["Event", "Cause", "District", "Subcounty", "Parish", "Location", "Deaths", "Duration (d)"]].notna().sum().to_string())

print(f"\ndistinct Event values: {df['Event'].unique().tolist()}")
print(f"rows per district:\n{df['District'].value_counts().to_string()}")

   Date (YMD)  Duration (d)   District  Subcounty Parish Location
0    1933/0/0           NaN     BUDUDA        NaN    NaN      NaN
1    1933/0/0           NaN     BUDUDA        NaN    NaN      NaN
2  2014/10/17           NaN  BULAMBULI     SISIYI    NaN      NaN
3    2018/6/7           NaN  BULAMBULI        NaN    NaN      NaN
4    2018/5/1           NaN   BUTALEJA        NaN    NaN      NaN
5    2014/6/4           NaN    SIRONKO      ZESUI    NaN      NaN
6    1991/9/2           NaN      MBALE    BUFUMBO    NaN      NaN
7  2013/10/25           NaN    SIRONKO     BUYOBO    NaN      NaN
8    2012/7/3           NaN     BUDUDA  BULUCHEKE    NaN      NaN
9    2002/6/1           NaN    SIRONKO   BUMASIFA    NaN      NaN

non-null counts for the key fields:
Event           271
Cause           228
District        271
Subcounty       149
Parish           43
Location         78
Deaths           17
Duration (d)     57

distinct Event values: ['FLOOD']
rows per district:
District
BUTALEJA     68

### Study area: seven districts, not nine

**BUKWO and KWEEN are excluded from the study area.** Two reasons, and the second
is the one that matters:

1. **They contribute almost nothing.** Across the whole inventory BUKWO has 5
   records and KWEEN 2, which survive cleaning as **1 and 2 district-days
   respectively**. A district-day classifier cannot learn anything from one
   positive example, and their presence adds two districts' worth of negative
   days against effectively no signal — which makes the class balance worse, not
   better.

2. **They are the wrong hazard.** Both sit high on the Mount Elgon massif, on the
   upper slopes and the Kenyan side of the caldera. At that elevation the
   rainfall-driven hazard is *landslide and debris flow*, not riverine flooding:
   there is little floodplain above them to inundate. The target variable here is
   flood occurrence, so records from districts whose wet-season hazard is
   predominantly mass movement are a different phenomenon wearing the same label.

DesInventar records both under the event type `FLOOD`, which is precisely the
problem — the inventory does not separate landslides from floods, so the
distinction has to be imposed geographically.

The lowland and mid-slope districts retained — Butaleja, Mbale, Manafwa, Bududa,
Sironko, Bulambuli, Kapchorwa — are where the Manafwa, Malaba and Sironko river
systems flood.

In [346]:
# High-massif districts whose wet-season hazard is mass movement rather than
# riverine flooding, and which contribute 1-2 district-days between them
EXCLUDED_DISTRICTS = ["BUKWO", "KWEEN"]

before = len(df)
district = df["District"].astype("string").str.strip().str.upper()
removed = df[district.isin(EXCLUDED_DISTRICTS)]

df = df[~district.isin(EXCLUDED_DISTRICTS)].reset_index(drop=True)

print(f"excluded {', '.join(EXCLUDED_DISTRICTS)}: {before} -> {len(df)} rows "
      f"({len(removed)} removed)")
print(f"\nrows removed per district:")
print(removed["District"].value_counts().to_string())
print(f"\nstudy area now {df['District'].nunique()} districts:")
print(", ".join(sorted(df["District"].unique())))

excluded BUKWO, KWEEN: 271 -> 264 rows (7 removed)

rows removed per district:
District
BUKWO    5
KWEEN    2

study area now 7 districts:
BUDUDA, BULAMBULI, BUTALEJA, KAPCHORWA, MANAFWA, MBALE, SIRONKO


## 2. Why EM-DAT is not used

EM-DAT (the Emergency Events Database, CRED/UCLouvain) covers Ugandan floods
through 2025 and was evaluated as a way to extend this record past DesInventar's
2018 cut-off. **It was merged, tested against daily rainfall, and rejected.** The
reasoning is kept here because excluding a major published inventory needs a
justification.

Seventeen EM-DAT rows named a Mount Elgon district, expanding to 49
district-level records. Three findings ruled them out:

1. **The unit of record does not match.** A DesInventar row is a district (often
   sub-county) office reporting a flood. An EM-DAT row is one *national* disaster
   event, with affected districts taken from its admin-unit list. Exploding that
   list gives a district-day row that **asserts** a flood rather than evidencing
   one, and the duration is the envelope of the national episode, not the length
   of any district's flood.

2. **The weaker labels dominated the training frame.** EM-DAT supplied 34% of
   events but **74% of daily rows**, because its episode envelopes ran to 78 days
   against DesInventar's median of 2. Capping EM-DAT spans at 2 days corrected the
   share to 32%, but only by discarding almost all of the duration information
   that was the reason for merging it.

3. **CHIRPS rainfall contradicted the attributions.** This was decisive. Of the
   six driest flood days in the merged set, two were EM-DAT attributions from the
   single 2019-12-18 eight-district event: KWEEN recorded 0.15 mm and BUKWO
   0.26 mm of rain over the preceding three days. Those districts were dry.
   The attribution came from EM-DAT's admin-unit list, not from any observation
   of flooding there.

A national inventory is the right tool for national statistics and the wrong tool
for district-day labels. Everything below therefore uses **DesInventar only**:
271 records, nine districts, 1933-2018.

The cost is real and should be stated: no labels after 2018, so the 2019-2025
rainfall record cannot be used for training. The extraction code has been removed;
reinstating EM-DAT would mean rebuilding it against `dataset/emdat-uganda.xlsx`,
which is retained.

## 3. Date validity

DesInventar pads unknown parts of a date with `0`, so `1933/0/0` is a year-only
record and `2013/10/0` is known only to the month. These are **incomplete rather
than corrupt** — the year is trustworthy, the finer parts were never recorded.

`classify_date` labels each entry's precision so the incomplete ones can be
counted before deciding what to do with them.

In [347]:
def classify_date(value) -> str:
    """Label the precision of a single `Date (YMD)` entry."""
    if pd.isna(value):
        return "missing"

    parts = str(value).split("/")
    if len(parts) != 3 or not all(part.strip().isdigit() for part in parts):
        return "malformed"

    year, month, day = (int(part) for part in parts)
    if month == 0 and day == 0:
        return "year only"
    if month == 0:
        return "day without month"
    if day == 0:
        return "month only"

    # Guards against real calendar errors such as 2013/2/30
    if pd.to_datetime(f"{year}/{month}/{day}", format="%Y/%m/%d", errors="coerce") is pd.NaT:
        return "impossible date"
    return "complete"


def invalid_date_report(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Rows whose date is not a complete calendar date, plus a category count."""
    categories = df[DATE_COLUMN].map(classify_date)
    invalid = df[categories != "complete"].copy()
    invalid["Date Issue"] = categories[categories != "complete"]
    return invalid, categories.value_counts()


invalid, counts = invalid_date_report(df)

print(f"{len(invalid)} of {len(df)} rows have an invalid/incomplete date\n")
print("Breakdown by category:")
print(counts.to_string())

print("\nDistinct invalid values:")
print(invalid[DATE_COLUMN].value_counts().to_string())

48 of 264 rows have an invalid/incomplete date

Breakdown by category:
Date (YMD)
complete             216
month only            42
year only              4
day without month      2

Distinct invalid values:
Date (YMD)
2007/9/0     12
2007/7/0     10
2010/2/0     10
1933/0/0      2
2013/11/0     2
2007/10/0     2
2011/2/0      2
2018/0/4      1
2013/10/0     1
2013/0/19     1
2012/0/0      1
1992/0/0      1
2010/4/0      1
2010/10/0     1
2011/7/0      1


### Findings — dates

**48 of 264 rows (18.2%) have an incomplete date.** None of the excluded BUKWO or
KWEEN records were among them, so the count is unchanged from the nine-district
inventory.

| Category | Rows | Meaning |
| --- | --- | --- |
| `month only` | 42 | day is `0` — e.g. `2007/9/0`, known to the month |
| `year only` | 4 | month and day both `0` — e.g. `1933/0/0` |
| `day without month` | 2 | month `0` but day set — `2013/0/19`, `2018/0/4` |

No rows are malformed, missing, or impossible — every entry is `YYYY/M/D`
shaped, so zero-padding is the only defect.

Two points that affect interpretation:

- **`2013/0/19` and `2018/0/4` are genuinely odd.** A known day with an unknown
  month is not a precision level; these two are likely data-entry errors.
- **The invalid dates cluster.** `2007/9/0` (12 rows), `2007/7/0` (10) and
  `2010/2/0` (10) account for 32 of the 48. These look like three large flood
  events recorded as many district-level rows sharing one imprecise date — so
  dropping them costs roughly **3 events, not 48 independent ones**. These three
  are the largest single loss in the pipeline and are worth naming in the
  methodology.

### Dropping the incomplete dates

Rows without a full calendar date cannot be aligned to daily rainfall or
soil-moisture observations, which is the purpose of this dataset here, so they
are removed. This is a deliberate trade: the three month-precision events above
are real floods, and they are being discarded for lack of a usable timestamp.

In [348]:
date_categories = df[DATE_COLUMN].map(classify_date)
dropped = int((date_categories != "complete").sum())

df = df[date_categories == "complete"].reset_index(drop=True)

print(f"dropped {dropped} rows with an invalid date")
print(f"df now holds {len(df)} rows ({len(df) / (len(df) + dropped):.1%} of the combined {len(df) + dropped})")
print("\nrows per district after the drop:")
print(df["District"].value_counts().to_string())

dropped 48 rows with an invalid date
df now holds 216 rows (81.8% of the combined 264)

rows per district after the drop:
District
BUTALEJA     56
SIRONKO      45
MANAFWA      35
BULAMBULI    27
BUDUDA       26
MBALE        17
KAPCHORWA    10


### Findings — effect of the drop

**216 rows retained (81.8%).** The loss is not geographically uniform:

| District | Before | After | Lost |
| --- | --- | --- | --- |
| BUTALEJA | 68 | 56 | 12 |
| SIRONKO | 58 | 45 | 13 |
| MANAFWA | 42 | 35 | 7 |
| BUDUDA | 35 | 26 | 9 |
| KAPCHORWA | 15 | 10 | 5 |
| BULAMBULI | 28 | 27 | 1 |
| MBALE | 18 | 17 | 1 |

SIRONKO and BUTALEJA absorb half the loss, while BULAMBULI and MBALE are barely
touched. This mildly biases any per-district event frequency and is worth stating
in the methodology.

## 4. Duplicate records

The dedup key must match the resolution the data will be used at. The target is a
**district-day flood prediction model**, where one training example answers "did
a flood occur in district D on day T". At that resolution two reports of the same
district-day are one example, however differently they were written up, so the
key is **date + district**.

An earlier version of this notebook keyed on date + district + sub-location
(`Subcounty`, `Parish`, `Location`). That was correct for a sub-location-level
analysis but wrong here: it left a single district-day split across up to 8 rows,
which would feed the model the same day many times over and inflate the positive
class.

Deliberately **excluded** from the key:

- **`Comments`, `Description of Cause`, `Source`** — these are the fields that
  vary between re-keyed entries of one event (typos, spacing, the source given as
  `District` / `district file` / `BUTALEJA`). Keying on them re-splits precisely
  the rows that need merging: the `2018/5/4` BUTALEJA district-day has 8 rows
  with 8 distinct `Comments` and 4 distinct `Source` values.
- **`Duration (d)`** — a defensible key component in principle, but it is null in
  76% of rows and, as it happens, never disagrees within a date+district group in
  this dataset, so it cannot discriminate anything here.
- **Sub-location** — see above. The detail is preserved as a count rather than
  thrown away, in `Affected Sub Locations`.

In [349]:
UNSPECIFIED = "~UNSPECIFIED~"

# Columns that hold no analytical payload, so ignoring them when comparing rows
PAYLOAD_EXCLUDED = ["Serial"]


def normalise(series: pd.Series) -> pd.Series:
    """Upper-case and trim a text column so casing/whitespace never split a group.

    Missing values become a sentinel rather than NaN, because NaN != NaN would
    otherwise stop two equally-unspecified values from grouping together.
    """
    return series.astype("string").str.strip().str.upper().fillna(UNSPECIFIED)


def place_key(df: pd.DataFrame) -> pd.Series:
    """Finest recorded sub-location within a district.

    No longer part of the duplicate key, but still needed to count how many
    distinct sub-locations a district-day affected. `Location` is included
    because roughly a third of rows leave `Subcounty` and `Parish` blank and name
    the sub-county in free text instead.
    """
    return (
        normalise(df["Subcounty"])
        + " | "
        + normalise(df["Parish"])
        + " | "
        + normalise(df["Location"])
    )


def duplicate_key(df: pd.DataFrame) -> pd.Series:
    """Identity of a record at the model's resolution: one district, one day."""
    return df[DATE_COLUMN].astype(str) + " | " + normalise(df["District"])

In [350]:
def duplicate_report(df: pd.DataFrame) -> dict[str, object]:
    """Rows reporting the same event on the same district-day."""
    group = duplicate_key(df)
    sizes = group.value_counts()
    repeated = sizes[sizes > 1]

    candidates = df[group.isin(repeated.index)].copy()
    candidates["Duplicate Key"] = group[group.isin(repeated.index)]

    payload = [c for c in df.columns if c not in PAYLOAD_EXCLUDED]

    return {
        "groups": len(repeated),
        "rows": int(repeated.sum()),
        # One row per group is the record to keep; the rest are the excess
        "redundant_rows": int(repeated.sum() - len(repeated)),
        "hard_duplicate_rows": int(df.duplicated(subset=payload, keep=False).sum()),
        "candidates": candidates.sort_values(["Duplicate Key", "Serial"]),
    }


dupes = duplicate_report(df)

print("Same event + date + district")
print(f"  duplicate groups:      {dupes['groups']}")
print(f"  rows involved:         {dupes['rows']}")
print(f"  redundant rows:        {dupes['redundant_rows']}  (dropping these keeps one per district-day)")
print(f"  identical but Serial:  {dupes['hard_duplicate_rows']}")
print(f"  distinct district-days: {duplicate_key(df).nunique()}")

# What the discarded sub-location key would have produced, for comparison
strict = df[DATE_COLUMN].astype(str) + " | " + normalise(df["District"]) + " | " + place_key(df)
print(f"\nkeying on date + district + sub-location instead would leave "
      f"{strict.nunique()} rows rather than {duplicate_key(df).nunique()}")

Same event + date + district
  duplicate groups:      31
  rows involved:         154
  redundant rows:        123  (dropping these keeps one per district-day)
  identical but Serial:  0
  distinct district-days: 93

keying on date + district + sub-location instead would leave 196 rows rather than 93


In [351]:
# Which columns actually disagree inside a district-day group? This decides the
# merge rule for each one.
candidates = dupes["candidates"]
conflicts = {
    column: int((candidates.groupby("Duplicate Key")[column].apply(lambda s: s.dropna().astype(str).nunique()) > 1).sum())
    for column in df.columns
}
print(f"columns with conflicting values inside a district-day (of {dupes['groups']} groups):")
for column, groups in sorted(conflicts.items(), key=lambda kv: -kv[1]):
    if groups:
        print(f"  {column:<24} {groups}")

print("\nthe only conflicting Cause values:")
cause_conflicts = candidates.groupby("Duplicate Key")["Cause"].apply(lambda s: s.dropna().unique())
for key, values in cause_conflicts[cause_conflicts.map(len) > 1].items():
    print(f"  {key:<26} {list(values)}")

print("\nDeaths reported more than once within a district-day:")
tolls = candidates.groupby("Duplicate Key")["Deaths"].agg(["count", "sum"])
print(tolls[tolls["count"] > 1].to_string())

columns with conflicting values inside a district-day (of 31 groups):
  Serial                   31
  Comments                 21
  Description of Cause     13
  Code Subcounty           13
  Subcounty                13
  Location                 11
  Source                   7
  Code Parish              6
  Parish                   6
  Cause                    2
  Deaths                   1

the only conflicting Cause values:
  2017/4/17 | BULAMBULI      ['Windstorm', 'Deforestation']
  2018/5/4 | BUTALEJA        ['Heavy Rain', 'El Niño']

Deaths reported more than once within a district-day:
                     count  sum
Duplicate Key                  
2011/9/16 | SIRONKO      2  7.0


### Findings — duplicates

**31 of the 93 district-days were reported more than once, accounting for 123
redundant rows.**

| Check | Result |
| --- | --- |
| Distinct district-days | 93 |
| District-days with >1 report | 31 |
| Rows involved | 154 |
| Redundant rows | 123 |
| Identical in every column but `Serial` | 0 |
| Rows the sub-location key would have left | 196 |

The last line is the point: the same 216 rows reduce to **93** at district-day
resolution but **196** if sub-location is in the key. The earlier key was leaving
a single district-day split up to 24 ways.

What actually conflicts inside a district-day, and so needs a merge rule:

| Column | Groups affected |
| --- | --- |
| `Comments` | 21 |
| `Description of Cause` | 13 |
| `Subcounty` / `Code Subcounty` | 13 |
| `Location` | 11 |
| `Source` | 7 |
| `Parish` / `Code Parish` | 6 |
| `Cause` | 2 |
| `Deaths` | 1 |

Two results make the merge safe:

- **`Cause` conflicts in only 2 of 31 groups** — `2017/4/17` BULAMBULI
  (Windstorm vs Deforestation) and `2018/5/4` BUTALEJA (Heavy Rain x6 vs
  El Nino). Only the first is a genuine tie; the second resolves cleanly to
  Heavy Rain on frequency.
- **`Deaths` is reported twice in only 1 group** (`2011/9/16` SIRONKO, 2 and 5),
  so summing to a district toll of 7 affects a single row.

`Duration (d)` does not appear in the conflict list at all — no district-day has
two different reported durations, so taking the maximum is unambiguous here.

### Collapsing to one row per district-day

One row survives per district-day. Each column is merged by a rule chosen from
what the diagnostic above shows actually conflicts:

| Column(s) | Rule | Why |
| --- | --- | --- |
| `Duration (d)` | maximum | Longest reported span, so the flood is labelled active for its full extent. Never actually conflicts in this data. |
| `Deaths`, `DataCards` | sum | Sub-location tolls add up to the district toll. Summing also preserves the grand total, which is checkable. |
| `Cause` | most frequent non-null | Nulls must not outvote a recorded cause. Ties break alphabetically so the result is deterministic. |
| `Comments`, `Description of Cause`, `Source` | longest | The most complete narrative, rather than whichever happened to sort first. |
| `Serial` | lowest | A stable event identifier for the daily expansion in section 5. |
| `Subcounty`, `Parish`, `Location` + codes | **dropped** | Meaningless once a row represents a whole district. Keeping one of eight sub-counties would misrepresent the row, so they are replaced by a count. |
| — | `Affected Sub Locations` | New column: how many distinct sub-locations the district-day affected. A severity proxy for the model, and the only part of the sub-location detail that survives. |

In [352]:
# Merge rules for the columns that conflict within a district-day
NUMERIC_AGGREGATION = {
    "Duration (d)": "max",  # longest reported span for the district-day
    "Deaths": "sum",        # per-sub-location tolls add up to the district toll
    "DataCards": "sum",
}
LONGEST_TEXT = ["Comments", "Description of Cause", "Source"]

# Dropped: a single district-day covers many sub-locations, so naming one of them
# would misrepresent the row. Replaced by `Affected Sub Locations`.
SUB_LOCATION_COLUMNS = ["Subcounty", "Code Subcounty", "Parish", "Code Parish", "Location"]


def collapse_group(group: pd.DataFrame) -> pd.Series:
    """Reduce one district-day to a single row."""
    row = group.iloc[0].copy()
    row["Serial"] = group["Serial"].min()

    for column, how in NUMERIC_AGGREGATION.items():
        # min_count keeps an all-null column null instead of turning it into 0
        row[column] = group[column].max() if how == "max" else group[column].sum(min_count=1)

    for column in LONGEST_TEXT:
        present = group[column].dropna().astype(str)
        if not present.empty:
            row[column] = max(present, key=len)

    causes = group["Cause"].dropna()
    if not causes.empty:
        # Most frequent cause, alphabetical on ties so re-runs agree
        row["Cause"] = min(causes.value_counts().items(), key=lambda kv: (-kv[1], kv[0]))[0]

    row["Affected Sub Locations"] = place_key(group).nunique()
    return row.drop(labels=SUB_LOCATION_COLUMNS)


def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """One row per district-day; single-report district-days pass through."""
    # Sorting by Serial makes the surviving row deterministic across runs
    ordered = df.assign(_group=duplicate_key(df)).sort_values(["_group", "Serial"])

    collapsed = pd.DataFrame(
        [collapse_group(rows) for _, rows in ordered.groupby("_group", sort=False)]
    ).drop(columns="_group")

    # Rebuilding from row Series loses the original dtypes, so restore the survivors
    retained = {c: t for c, t in df.dtypes.items() if c not in SUB_LOCATION_COLUMNS}
    return collapsed.astype(retained).reset_index(drop=True)


before = len(df)
deaths_before = df["Deaths"].sum()
cards_before = df["DataCards"].sum()

df = collapse_duplicates(df)

print(f"{before} -> {len(df)} rows ({before - len(df)} redundant rows removed)")

print("\ndistrict-days affecting more than one sub-location:")
multi = df[df["Affected Sub Locations"] > 1]
print(multi[[DATE_COLUMN, "District", "Affected Sub Locations", "Duration (d)", "Deaths"]].to_string(index=False))

# Summed columns must preserve their totals, and the key must now be unique
print(f"\nDeaths total preserved:      {df['Deaths'].sum() == deaths_before} ({df['Deaths'].sum():.0f})")
print(f"DataCards total preserved:   {df['DataCards'].sum() == cards_before} ({df['DataCards'].sum()})")
print(f"remaining duplicate groups:  {duplicate_report(df)['groups']}")
print(f"one row per district-day:    {duplicate_key(df).is_unique}")

216 -> 93 rows (123 redundant rows removed)

district-days affecting more than one sub-location:
Date (YMD)  District  Affected Sub Locations  Duration (d)  Deaths
 2005/5/16     MBALE                       2         150.0     NaN
2006/11/28  BUTALEJA                       2           NaN     NaN
 2007/10/1    BUDUDA                       8           NaN     2.0
  2007/8/8   SIRONKO                       2           NaN     NaN
  2009/4/6     MBALE                       2           2.0     NaN
 2010/10/4 BULAMBULI                       8          30.0     NaN
 2010/2/20  BUTALEJA                       2           NaN     NaN
 2010/3/10   MANAFWA                      24          14.0     NaN
 2010/5/14   SIRONKO                       5           NaN     NaN
 2011/9/15 BULAMBULI                       7           3.0     NaN
 2011/9/15   SIRONKO                       7           NaN     2.0
 2011/9/16   SIRONKO                       4           NaN     7.0
 2011/9/20  BUTALEJA            

### Findings — after collapsing

**216 -> 93 rows**, one per district-day. The re-check reports 0 duplicate groups
and the key is unique. `Deaths` still totals 34 and `DataCards` still totals 216,
which is the check that summing moved values rather than losing them.

**27 of the 93 district-days affected more than one sub-location**, up to 24 for
`2010/3/10` MANAFWA. That count is now the `Affected Sub Locations` column — a
plausible severity feature, since a flood reaching 24 sub-counties is a larger
event than one reaching one, and it is the only trace of the sub-location detail
that survives the merge.

Note the count is distinct *sub-locations*, not reports merged: `2018/5/4`
BUTALEJA was built from 8 reports but only 3 distinct sub-locations, because 6 of
those reports named the same sub-county.

**Trade-off accepted here:** `Subcounty`, `Parish`, `Location` and their code
columns are dropped. Any later sub-district analysis must go back to the raw
file — this frame cannot answer sub-county questions.

## 5. Multi-district storms

One weather system hitting several neighbouring districts is recorded as
separate district rows sharing a date, so candidate storms are found by grouping
on date (+ `Cause`) and keeping the groups that span more than one district.

`Cause` does little separating work here — it is `Heavy Rain` in the large
majority of rows with a number of nulls — so the date does most of the grouping.

In [353]:
def storm_clusters(df: pd.DataFrame, use_cause: bool = True) -> pd.DataFrame:
    """Group rows into candidate storms and keep those touching >1 district."""
    key = df[DATE_COLUMN].astype(str)
    if use_cause:
        key = key + " | " + normalise(df["Cause"])

    grouped = pd.DataFrame({"key": key, "district": normalise(df["District"])}).groupby("key")
    clusters = grouped["district"].agg(
        districts="nunique",
        rows="size",
        district_names=lambda s: ", ".join(sorted(set(s))),
    )
    return clusters[clusters["districts"] > 1].sort_values("districts", ascending=False)


# Every remaining row has a complete date, so same-date grouping means same *day*
# rather than, as it would for entries like "2007/9/0", merely the same month.
for key_label, use_cause in [("date + cause", True), ("date only", False)]:
    clusters = storm_clusters(df, use_cause=use_cause)
    print(
        f"{key_label:<13} {len(clusters):>2} multi-district clusters, "
        f"{clusters['rows'].sum():>3} rows, max {clusters['districts'].max()} districts"
    )

print("\nMulti-district clusters on date + cause:")
print(storm_clusters(df).to_string())

date + cause   7 multi-district clusters,  14 rows, max 2 districts
date only      8 multi-district clusters,  16 rows, max 2 districts

Multi-district clusters on date + cause:
                        districts  rows      district_names
key                                                        
2006/8/5 | HEAVY RAIN           2     2  KAPCHORWA, SIRONKO
2008/6/2 | HEAVY RAIN           2     2    MANAFWA, SIRONKO
2010/3/10 | HEAVY RAIN          2     2   BUTALEJA, MANAFWA
2010/3/4 | HEAVY RAIN           2     2     BUTALEJA, MBALE
2010/4/28 | HEAVY RAIN          2     2     BUDUDA, MANAFWA
2010/5/25 | HEAVY RAIN          2     2      MBALE, SIRONKO
2011/9/15 | HEAVY RAIN          2     2  BULAMBULI, SIRONKO


### Findings — storms

**7 multi-district clusters on date + cause** (8 on date alone), covering 14
rows, with a maximum footprint of **2 districts**.

Every cluster is a pair of neighbouring districts, which is what a real weather
footprint should look like: BUTALEJA–MANAFWA, BUTALEJA–MBALE, KAPCHORWA–SIRONKO,
BULAMBULI–SIRONKO, MBALE–SIRONKO, MANAFWA–SIRONKO, BUDUDA–MANAFWA. All seven are
`Heavy Rain`.

Two things make this a stronger result than it first appears:

- **The date cleaning in section 3 mattered.** On the raw data this check found
  clusters spanning 4 districts, but the 4-district cluster was `2007/7/0`,
  meaning "some time in July 2007" — evidence of a shared *month*, not a shared
  storm. Once imprecise dates are removed, nothing exceeds two districts.
- **These are independently reported.** Each district in a cluster filed its own
  report on the same day; the co-occurrence is observed rather than asserted by a
  central agency. That is exactly the property EM-DAT lacked (section 2), and it
  is why these 7 clusters are trustworthy where EM-DAT's 8-district footprints
  were not.

**Caveat.** Clustering uses the reported start date only, so a storm spanning a
date boundary will not cluster under exact-date equality. A proper footprint
would need an interval-overlap test using `Date + Duration`; 7 is a lower bound.

## 6. Expanding events to their full day span

Each row so far is one district-day *report*, dated on the day the flood began. A
flood with `Duration (d) = 14` in fact affected its district on 14 consecutive
days, and rainfall or soil-moisture observations exist for every one of those
days. To join against daily data, each event is expanded into one row per day it
was active.

Conventions used, and why:

- **`Duration (d) = N` spans N days**, from the reported date through
  `date + (N - 1)` inclusive — DesInventar defines the field as how long the
  event lasted, so a 1-day event must not become two rows.
- **A missing duration counts as 1 day.** Most events leave the
  field blank, and treating those as single-day events keeps every event in the
  output rather than discarding most of the inventory.
- **Durations are capped at 7 days.** See below.

### Why cap at 7 days

A reported duration is the length of the *flood episode*, and expanding it treats
every day of that episode as a day the district was flooded. That assumption
holds for a few days and then stops holding, which CHIRPS can be used to show
directly.

Mean CHIRPS rainfall on the expanded days, against the regional baseline of
**4.82 mm/day** across all 92,043 district-days:

| Days of span | Rows | Events | Mean rainfall | vs baseline |
| --- | --- | --- | --- | --- |
| 1–3 | 112 | 94 | 8.00 mm | **1.66x** |
| 4–7 | 26 | 7 | 6.45 mm | **1.34x** |
| 8–14 | 29 | 5 | 4.87 mm | **1.01x** |
| 15–21 | 21 | 3 | 9.37 mm | 1.94x |

**Days 8–14 of an expanded span are indistinguishable from an average day in the
region** — 4.87 mm against a 4.82 mm baseline, a ratio of 1.01. Whatever those
rows are, they carry no rainfall signal that separates them from the negative
class, so labelling them as flood days adds noise to both classes at once. Days
1–3 (1.66x) and 4–7 (1.34x) are clearly elevated. **The signal is gone by day 8**,
which is where the cap belongs.

The 15–21 band looks wet, but that figure rests on **3 events** — the 150-day and
two 30-day records. It is not evidence that long spans are meaningful; it is
evidence that three records dominate the tail. At a 21-day cap those three
supplied 63 of 190 rows, a third of the frame, from 3% of the events.

#### The literature anchor

The cap value is not chosen freely. Najibi & Devineni (2018) classify global flood
events by duration into three bands, derived from the Dartmouth Flood Observatory
global inventory:

| Band | Duration |
| --- | --- |
| short | 1–7 days |
| moderate | 8–20 days |
| long | 21 days and above |

> Najibi, N. and Devineni, N.: *Recent trends in the frequency and duration of
> global floods*, Earth Syst. Dynam., **9**, 757–783, 2018.
> https://esd.copernicus.org/articles/9/757/2018/

**7 days is the upper bound of the `short` band**, so the cap has a published
meaning: no event is expanded beyond the duration of a short flood. That the
CHIRPS signal disappears at day 8 and the `short`/`moderate` boundary falls at
day 8 is a coincidence, but a useful one — the empirical cut-off and the
taxonomy's cut-off agree, so the cap is defensible on either ground
independently.

The same taxonomy supplies `Duration Class`, which is computed from the
*reported, uncapped* duration. A 150-day record is therefore still labelled
`long` even though it now spans 7 rows.

**To be clear about the citation:** Najibi & Devineni classify durations, they do
not truncate them. The cap is a modelling decision taken here, informed by their
bands; it is not a method drawn from the paper.

**Unlike the previous 21-day cap, this one is not band-preserving.** 21 sat exactly
on the moderate/long boundary, so no event changed band; 7 sits on the short/moderate
boundary, so a 150-day event now spans a `short` number of days while
`Duration Class` still records it as `long`. The reported value stays in
`Duration (d)` and the band in `Duration Class`, so nothing is lost — but the span
and the band no longer agree, and that is deliberate.

> The expansion deliberately produces rows that repeat `Serial`, `District` and
> the narrative fields. These are **not** duplicates: each row is a distinct
> *(event, day)* observation, uniquely identified by `Serial` + `Day Index`.

In [354]:
# Najibi & Devineni (2018) duration bands: short 1-7, moderate 8-20, long 21+
DURATION_BANDS = [(7, "short"), (20, "moderate")]

# 7 days: the upper bound of the "short" band, and the point at which CHIRPS
# rainfall on expanded days falls to the regional baseline - see the discussion
# above. Note this is NOT band-preserving, unlike the earlier 21-day cap.
MAX_EVENT_DAYS = 7


def classify_duration(value) -> str:
    """Band a reported duration, keeping unrecorded durations distinguishable.

    Nulls are `unknown` rather than `short`: those events never recorded a
    duration, which is a reporting artefact and should not be conflated with the
    events genuinely reported as lasting one day.
    """
    if pd.isna(value):
        return "unknown"
    for upper, label in DURATION_BANDS:
        if value <= upper:
            return label
    return "long"


def expand_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """One row per day each event was active.

    Adds `Duration Class` (from the reported duration, uncapped), `Observation
    Date` (the calendar day, as a datetime for joining to daily
    rainfall/soil-moisture series), `Day Index` (1-based position within the
    event) and `Event Days` (the capped span actually used).
    """
    days = df["Duration (d)"].fillna(1).clip(lower=1, upper=MAX_EVENT_DAYS).astype(int)

    base = df.reset_index(drop=True).assign(
        **{
            "Duration Class": df["Duration (d)"].map(classify_duration).to_numpy(),
            "Event Days": days.to_numpy(),
        }
    )

    expanded = base.loc[base.index.repeat(base["Event Days"])].copy()
    # cumcount runs within each original row, so it numbers the days of one event
    expanded["Day Index"] = expanded.groupby(level=0).cumcount() + 1
    expanded["Observation Date"] = pd.to_datetime(
        expanded[DATE_COLUMN], format="%Y/%m/%d"
    ) + pd.to_timedelta(expanded["Day Index"] - 1, unit="D")

    return expanded.reset_index(drop=True)


events = len(df)
uncapped = int(df["Duration (d)"].fillna(1).clip(lower=1).sum())
shortened = df[df["Duration (d)"] > MAX_EVENT_DAYS]

df = expand_to_daily(df)

print(f"expanded {events} events -> {len(df)} rows")
print(f"total rows in df: {len(df)}")
print(f"\nuncapped this would be {uncapped} rows; the {MAX_EVENT_DAYS}-day cap removes "
      f"{uncapped - len(df)}")
print(f"{len(shortened)} events were shortened by the cap")

print(f"\ndistinct calendar days covered: {df['Observation Date'].nunique()}")
print(f"date range: {df['Observation Date'].min():%Y-%m-%d} -> {df['Observation Date'].max():%Y-%m-%d}")

# Serial + Day Index must identify a row uniquely, or the expansion double-counted
print(f"Serial + Day Index unique: {df.groupby(['Serial', 'Day Index']).ngroups == len(df)}")
print(f"Event Days sums to row count: {int(df.groupby('Serial')['Event Days'].first().sum()) == len(df)}")

# The 7-day cap deliberately moves long events onto a short span, so bands are
# reported rather than asserted. `Duration Class` keeps the reported band.
recorded = df.dropna(subset=["Duration (d)"]).drop_duplicates("Serial")
rebanded = recorded[
    recorded["Duration (d)"].map(classify_duration) != recorded["Event Days"].map(classify_duration)
]
print(f"{len(rebanded)} of {len(recorded)} dated events now span fewer days than their "
      f"reported band, by design:")
print(rebanded[[DATE_COLUMN, "District", "Duration (d)", "Duration Class", "Event Days"]].to_string(index=False))

expanded 93 events -> 137 rows
total rows in df: 137

uncapped this would be 334 rows; the 7-day cap removes 197
5 events were shortened by the cap

distinct calendar days covered: 120
date range: 1991-09-02 -> 2018-06-20
Serial + Day Index unique: True
Event Days sums to row count: True
5 of 14 dated events now span fewer days than their reported band, by design:
Date (YMD)  District  Duration (d) Duration Class  Event Days
 2005/5/16     MBALE         150.0           long           7
 2010/10/4 BULAMBULI          30.0           long           7
 2010/3/10   MANAFWA          14.0       moderate           7
  2010/3/3     MBALE           8.0       moderate           7
  2010/3/4     MBALE          30.0           long           7


In [355]:
print("events and rows per duration band:")
by_band = df.groupby("Duration Class").agg(events=("Serial", "nunique"), rows=("Serial", "size"))
print(by_band.reindex(["unknown", "short", "moderate", "long"]).dropna(how="all").to_string())

print("\nrows contributed by each reported duration:")
contribution = df.groupby(df["Duration (d)"].fillna(0)).agg(events=("Serial", "nunique"), rows=("Serial", "size"))
contribution.index = contribution.index.map(lambda d: "not recorded" if d == 0 else f"{int(d)} days")
print(contribution.to_string())

print("\nrows per district, before vs after the expansion:")
comparison = pd.DataFrame(
    {
        "events": df.groupby("District")["Serial"].nunique(),
        "day_rows": df["District"].value_counts(),
    }
).sort_values("day_rows", ascending=False)
print(comparison.to_string())

# Distinct events overlapping on the same district-day: expected, not duplication
district_days = df.groupby(["District", "Observation Date"])["Serial"].nunique()
print(f"\ndistrict-day combinations:                 {len(district_days)}")
print(f"...drawing on more than one event record:  {int((district_days > 1).sum())}")

print("\nexample - one event unrolled across its span:")
longest = df.loc[df["Event Days"].idxmax(), "Serial"]
sample = df[df["Serial"] == longest]
print(sample[["Serial", DATE_COLUMN, "Duration (d)", "Duration Class", "Event Days", "Day Index", "Observation Date"]].head(4).to_string(index=False))
print(f"... {len(sample)} rows in total for Serial {longest}")

events and rows per duration band:
                events  rows
Duration Class              
unknown             79    79
short                9    23
moderate             2    14
long                 3    21

rows contributed by each reported duration:
              events  rows
Duration (d)              
not recorded      79    79
1 days             4     4
2 days             2     4
3 days             1     3
6 days             2    12
8 days             1     7
14 days            1     7
30 days            2    14
150 days           1     7

rows per district, before vs after the expansion:
           events  day_rows
District                   
MBALE          14        33
BUTALEJA       24        25
BUDUDA         11        21
SIRONKO        18        18
BULAMBULI       9        17
MANAFWA         7        13
KAPCHORWA      10        10

district-day combinations:                 131
...drawing on more than one event record:  6

example - one event unrolled across its span:
 Seria

### Findings — daily expansion

**93 district-day events expand to 137 daily rows**, covering 120 distinct
calendar days between 1991-09-02 and 2018-06-20. `Serial` + `Day Index` is unique
across all 137 rows and the per-event spans sum back to the row count, so no day
is double-counted.

**The 7-day cap removes 197 rows** — uncapped the frame would be 334, so the cap
now discards more rows than it keeps. Five events were shortened:

| Date | District | Reported | Band | Span |
| --- | --- | --- | --- | --- |
| 2005/5/16 | MBALE | 150 days | long | 7 |
| 2010/3/4 | MBALE | 30 days | long | 7 |
| 2010/10/4 | BULAMBULI | 30 days | long | 7 |
| 2010/3/10 | MANAFWA | 14 days | moderate | 7 |
| 2010/3/3 | MBALE | 8 days | moderate | 7 |

All five now span fewer days than their reported band implies. That is the
intended effect, and `Duration Class` still carries the reported band so the
information is not lost.

Rows by duration band, computed from the *reported* duration:

| Band | Events | Rows |
| --- | --- | --- |
| unknown (not recorded) | 79 | 79 |
| short (1–7 days) | 9 | 23 |
| moderate (8–20 days) | 2 | 14 |
| long (21+ days) | 3 | 21 |

**85% of events (79 of 93) never recorded a duration** and contribute a single day
each. The three long-duration events now supply 21 rows rather than the 63 they
did at a 21-day cap — down from a third of the frame to 15%.

Rows per district:

| District | Events | Day rows |
| --- | --- | --- |
| MBALE | 14 | 33 |
| BUTALEJA | 24 | 25 |
| BUDUDA | 11 | 21 |
| SIRONKO | 18 | 18 |
| BULAMBULI | 9 | 17 |
| MANAFWA | 7 | 13 |
| KAPCHORWA | 10 | 10 |

**The 7-day cap markedly improved the district balance.** At 21 days MBALE had 62
day rows against BUTALEJA's 25, a 2.5x gap driven by one 150-day record; it is now
33 against 25. Day-row counts still partly track duration-reporting completeness
rather than flood frequency — BUTALEJA has the most events (24) but 25 day rows
because almost none of its reports state a duration — but no single record
dominates any longer, and the thinnest district now has 10 rows rather than 1.

Only 6 of 131 district-day combinations draw on more than one event record.

## 7. Result

`df` holds **137 daily rows** describing 93 district-day flood events across seven
districts, drawn from 271 DesInventar records:

| Stage | Rows | Change |
| --- | --- | --- |
| Loaded | 271 | — |
| Restricted to the study area | 264 | −7 BUKWO / KWEEN |
| Complete dates only | 216 | −48 incomplete dates |
| Collapsed to one row per district-day | 93 | −123 redundant reports |
| Expanded to day spans, capped at 7 days | 137 | +44 event-days |

Each row is one *(district-day event, day)* pair, uniquely keyed by `Serial` +
`Day Index`, carrying:

- `Observation Date` — datetime, ready to join against daily rainfall and
  soil-moisture series
- `Duration Class` — Najibi & Devineni band from the reported, uncapped duration
  (`unknown` / `short` / `moderate` / `long`)
- `Event Days` — the capped span actually expanded
- `Duration (d)` — the duration as reported, so the cap is reversible
- `Affected Sub Locations` — sub-location count, a severity proxy

Known limitations to carry into the analysis:

1. **The record stops in 2018.** DesInventar has no rows after `2018/6/20`, so
   rainfall and soil-moisture data for 2019–2025 cannot be used for training.
   This is the price of excluding EM-DAT (section 2) and is the single biggest
   constraint on the dataset.
2. **137 positives is a small training set**, and the 7-day cap reduced it from
   190. The cap is justified by the rainfall evidence in section 6, but the trade
   against sample size is real and should be acknowledged.
3. **85% of events (79 of 93) have no recorded duration** and count as a single
   day. Day-row counts therefore partly measure reporting completeness rather
   than flood extent.
4. Three flood events (`2007/7/0`, `2007/9/0`, `2010/2/0`) were dropped for
   month-only dates, accounting for 32 of the 48 removed rows. Date-filter loss
   is uneven across districts, falling mostly on SIRONKO and BUTALEJA.
5. **The 7-day cap is not band-preserving.** Five events span fewer days than
   their reported `Duration Class`. This is deliberate and documented, but it
   means span and band disagree for those records.
6. **BUKWO and KWEEN are excluded on a hazard argument, not a measurement.** The
   claim that their wet-season hazard is mass movement rather than riverine
   flooding is geographic reasoning, not something tested against this data —
   DesInventar labels both `FLOOD`. Their 7 records are recoverable by removing
   one filter if that judgement is revisited.
7. **The same landslide/flood conflation may persist in BUDUDA**, which is the
   most landslide-prone district in Uganda and is retained. Filtering on `Cause`
   rather than geography would address this more precisely.
8. **Some reported flood days have almost no antecedent rainfall in CHIRPS** —
   `2016-01-04` SIRONKO and `2016-11-14` BUTALEJA record zero rain over the
   preceding three days. Candidate label errors or non-rainfall triggers.
9. `Subcounty`, `Parish` and `Location` are dropped by the district-day merge;
   sub-district questions need the raw file.
10. `Cause` was resolved by majority vote in the 2 district-days where reports
    disagreed, so `2017/4/17` BULAMBULI carries an arbitrary tie-break.
11. `Deaths` is present in only 17 of the original rows, so impact severity
    cannot be modelled from this inventory alone.

## 8. Export

Writes the analysis-ready frame to `dataset/flood_labelled_data.csv`.

`Serial`, `Date (YMD)` and `Day Index` are included alongside the per-day
`Observation Date` so downstream work can recover the **event onset** exactly:
`Day Index == 1` selects one row per event. Reconstructing onsets from
consecutive-day runs instead loses events that share a district-day or fall on
adjacent days, so the identifiers are carried rather than inferred.

The path is resolved against `REPO_ROOT`, so the file lands in `dataset/`
regardless of the kernel's working directory.

`Duration (d)` is exported with unrecorded durations filled to **1**, matching the
single day such an event is expanded to, and cast to integer. `Duration Class`
still separates a genuinely reported 1-day flood (`short`) from one whose duration
was never recorded (`unknown`), so filling the column loses nothing.

In [356]:
# Generate CSV from data

EXPORT_COLUMNS = [
    # Event identity, so `Day Index == 1` recovers onsets without inference
    "Serial",
    DATE_COLUMN,
    "Day Index",
    # The per-day date to join daily rainfall / soil moisture on
    "Observation Date",
    "District",
    "Duration (d)",
    "Event Days",
    "Duration Class",
]

EXPORT_PATH = REPO_ROOT / "dataset/flood_labelled_data.csv"

minimal_df = df[EXPORT_COLUMNS].sort_values(["Observation Date", "District"]).copy()

# An unrecorded duration is expanded as a single day, so the exported column says
# so outright rather than leaving a null the reader has to interpret. With no
# nulls left the column becomes an integer, so spans read as 7 rather than 7.0.
# The distinction between "reported as 1 day" and "never recorded" is not lost -
# it survives in `Duration Class` as `short` versus `unknown`.
minimal_df["Duration (d)"] = minimal_df["Duration (d)"].fillna(1).astype(int)

minimal_df.to_csv(EXPORT_PATH, index=False)

print(f"wrote {len(minimal_df)} rows to {EXPORT_PATH}")
print(f"  events (Day Index == 1): {int((minimal_df['Day Index'] == 1).sum())}")
print(f"  date range: {minimal_df['Observation Date'].min():%Y-%m-%d} -> {minimal_df['Observation Date'].max():%Y-%m-%d}")
print(f"  rows with no reported duration, filled to 1: "
      f"{int((minimal_df['Duration Class'] == 'unknown').sum())}")
print()
print(minimal_df.head(8).to_string(index=False))

wrote 137 rows to /Users/morenikeji/Desktop/SchoolWork/msc_advanced_computer_science/msc_project/aquavates/dataset/flood_labelled_data.csv
  events (Day Index == 1): 93
  date range: 1991-09-02 -> 2018-06-20
  rows with no reported duration, filled to 1: 79

 Serial Date (YMD)  Day Index Observation Date  District  Duration (d)  Event Days Duration Class
   6585   1991/9/2          1       1991-09-02     MBALE             1           1        unknown
   6288   1994/5/4          1       1994-05-04 KAPCHORWA             1           1        unknown
   6214   1998/3/8          1       1998-03-08     MBALE             1           1        unknown
    629  2001/10/1          1       2001-10-01    BUDUDA             1           1        unknown
   6325   2002/6/1          1       2002-06-01   SIRONKO             1           1        unknown
   4736   2003/9/1          1       2003-09-01 KAPCHORWA             1           1        unknown
   5097  2004/5/11          1       2004-05-11     MBAL